# 04 — Chronological Modeling and Hyperparameter Tuning

## Objectives

This notebook uses the frozen Notebook 03 feature handoff to select and freeze one model for next-day unhealthy-air classification.

It will:

- verify the 6,000-row modeling dataset and its 65-feature manifest;
- freeze common chronological train, validation, and test periods;
- keep every test label outside model and threshold selection;
- evaluate persistence before machine learning;
- tune Logistic Regression, Random Forest, and Histogram Gradient Boosting using expanding-window cross-validation on Dhaka training data;
- select one model using Dhaka validation PR-AUC;
- select a recall-oriented threshold using Dhaka validation F2;
- calculate validation-only permutation importance;
- refit the frozen model on Dhaka train plus validation data;
- save the model, split assignments, validation evidence, and Notebook 05 protocol.

The other four cities remain unseen transfer locations during development. This notebook does **not** score the final test period or perform Notebook 05 error analysis.


## 1. Environment and Shared-Storage Setup

The notebook reads Notebook 03 outputs directly from the shared Google Drive folder and writes modeling artifacts to the same project folder. It does not clone or update the GitHub repository inside Colab.

For a non-Colab runtime, set `CSE437_STORAGE_ROOT` to the folder containing the project's `processed`, `models`, and `figures` directories.


In [ ]:
from pathlib import Path
from IPython.display import display

import hashlib
import json
import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path(
        "/content/drive/MyDrive/CSE437_air_quality_group_18"
    )
else:
    STORAGE_ROOT = Path(
        os.environ.get("CSE437_STORAGE_ROOT", ".")
    ).resolve()

PATHS = {
    "processed": STORAGE_ROOT / "processed",
    "models": STORAGE_ROOT / "models" / "notebook_04",
    "figures": STORAGE_ROOT / "figures" / "notebook_04",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_JOBS = int(os.environ.get("CSE437_N_JOBS", "-1"))
np.random.seed(RANDOM_STATE)

SELECTED_CITIES = [
    "Dhaka",
    "Dinājpur",
    "Bherāmāra",
    "Bhola",
    "Cox’s Bāzār",
]

SOURCE_CITY = "Dhaka"
TARGET_COLUMN = "next_day_unhealthy"

INPUT_FILE = PATHS["processed"] / "modeling_dataset.csv"
FEATURE_MANIFEST_FILE = (
    PATHS["processed"] / "notebook_03_feature_manifest.csv"
)
FEATURE_SUMMARY_FILE = (
    PATHS["processed"] / "notebook_03_feature_summary.json"
)

SPLIT_ASSIGNMENTS_FILE = (
    PATHS["processed"] / "notebook_04_split_assignments.csv"
)
CV_RESULTS_FILE = (
    PATHS["processed"] / "notebook_04_cv_results.csv"
)
VALIDATION_RESULTS_FILE = (
    PATHS["processed"] / "notebook_04_validation_results.csv"
)
FEATURE_IMPORTANCE_FILE = (
    PATHS["processed"]
    / "notebook_04_validation_feature_importance.csv"
)
MODELING_SUMMARY_FILE = (
    PATHS["processed"] / "notebook_04_modeling_summary.json"
)

FINAL_MODEL_FILE = PATHS["models"] / "final_model.joblib"
FINAL_PROTOCOL_FILE = (
    PATHS["models"] / "final_model_protocol.json"
)
PR_CURVE_FILE = PATHS["figures"] / "validation_pr_curves.png"

print("Storage root:", STORAGE_ROOT)
print("Modeling input:", INPUT_FILE)
print("Model output folder:", PATHS["models"])
print("Figure output folder:", PATHS["figures"])
print("Parallel jobs:", N_JOBS)


## 2. Load and Verify the Frozen Notebook 03 Handoff

Notebook 04 must use the saved feature list without reconstructing targets, adding same-day information, or silently changing feature windows.

The following checks confirm dimensions, column roles, city coverage, exact next-calendar-day alignment, feature-manifest order, and numeric completeness. Target values are checked only for structural validity; no test-period prevalence or performance is calculated.


In [ ]:
def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


required_input_files = [
    INPUT_FILE,
    FEATURE_MANIFEST_FILE,
    FEATURE_SUMMARY_FILE,
]

for required_file in required_input_files:
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required Notebook 03 output was not found: {required_file}"
        )

with FEATURE_SUMMARY_FILE.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

modeling = pd.read_csv(
    INPUT_FILE,
    parse_dates=["feature_date", "target_date"],
)

feature_manifest = pd.read_csv(FEATURE_MANIFEST_FILE)

identifier_columns = feature_summary["identifier_columns"]
outcome_columns = feature_summary["outcome_columns"]
feature_columns = feature_summary["feature_columns"]

assert modeling.shape == (6000, 71)
assert feature_summary["output_rows"] == 6000
assert feature_summary["output_rows_per_city"] == 1200
assert feature_summary["selected_cities"] == SELECTED_CITIES
assert identifier_columns == [
    "city",
    "feature_date",
    "target_date",
]
assert outcome_columns == [
    "target_daily_aqi",
    "next_day_unhealthy",
    "next_day_usg_or_worse",
]
assert len(feature_columns) == 65
assert list(modeling.columns) == [
    *identifier_columns,
    *outcome_columns,
    *feature_columns,
]
assert feature_manifest["feature"].tolist() == feature_columns
assert feature_manifest["nearest_day_before_target"].ge(1).all()
assert set(identifier_columns).isdisjoint(feature_columns)
assert set(outcome_columns).isdisjoint(feature_columns)

assert set(modeling["city"]) == set(SELECTED_CITIES)
assert not modeling.duplicated(["city", "target_date"]).any()
assert (
    modeling["target_date"] - modeling["feature_date"]
).eq(pd.Timedelta(days=1)).all()
assert modeling[feature_columns].notna().all().all()
assert modeling[feature_columns].apply(
    pd.api.types.is_numeric_dtype
).all()
assert np.isfinite(
    modeling[feature_columns].to_numpy(dtype=float)
).all()
assert set(modeling[TARGET_COLUMN].unique()).issubset({0, 1})

modeling = (
    modeling.sort_values(["city", "target_date"])
    .reset_index(drop=True)
)

expected_calendar = pd.date_range(
    "2022-08-12",
    "2025-11-23",
    freq="D",
)

for city in SELECTED_CITIES:
    city_dates = modeling.loc[
        modeling["city"].eq(city),
        "target_date",
    ].tolist()

    assert city_dates == list(expected_calendar)

input_checksums = {
    INPUT_FILE.name: calculate_sha256(INPUT_FILE),
    FEATURE_MANIFEST_FILE.name: calculate_sha256(
        FEATURE_MANIFEST_FILE
    ),
    FEATURE_SUMMARY_FILE.name: calculate_sha256(
        FEATURE_SUMMARY_FILE
    ),
}

input_audit = pd.DataFrame({
    "item": [
        "Modeling rows",
        "Columns",
        "Cities",
        "Rows per city",
        "Historical predictors",
        "Earliest target date",
        "Latest target date",
    ],
    "verified_value": [
        len(modeling),
        modeling.shape[1],
        modeling["city"].nunique(),
        modeling.groupby("city").size().min(),
        len(feature_columns),
        modeling["target_date"].min().date(),
        modeling["target_date"].max().date(),
    ],
})

display(input_audit)
display(feature_manifest.head())

print("Notebook 03 modeling handoff verified.")
print("Modeling SHA-256:", input_checksums[INPUT_FILE.name])


## 3. Freeze Common Chronological Splits

All five cities use identical target-date boundaries. The 1,200 target dates per city are divided into 70% training, 15% validation, and 15% test periods:

| Split | Target dates | Rows per city |
| --- | --- | ---: |
| Train | 12 August 2022–28 November 2024 | 840 |
| Validation | 29 November 2024–27 May 2025 | 180 |
| Test | 28 May 2025–23 November 2025 | 180 |

Historical features at a boundary are valid because every predictor was already verified to originate at least one day before its target. Split membership is based on `target_date`, never random row assignment.


In [ ]:
TRAIN_START = pd.Timestamp("2022-08-12")
TRAIN_END = pd.Timestamp("2024-11-28")
VALIDATION_START = pd.Timestamp("2024-11-29")
VALIDATION_END = pd.Timestamp("2025-05-27")
TEST_START = pd.Timestamp("2025-05-28")
TEST_END = pd.Timestamp("2025-11-23")

split_conditions = [
    modeling["target_date"].between(TRAIN_START, TRAIN_END),
    modeling["target_date"].between(
        VALIDATION_START,
        VALIDATION_END,
    ),
    modeling["target_date"].between(TEST_START, TEST_END),
]

modeling["split"] = np.select(
    split_conditions,
    ["train", "validation", "test"],
    default="outside_frozen_period",
)

assert not modeling["split"].eq("outside_frozen_period").any()
assert TRAIN_END < VALIDATION_START
assert VALIDATION_END < TEST_START
assert TRAIN_END + pd.Timedelta(days=1) == VALIDATION_START
assert VALIDATION_END + pd.Timedelta(days=1) == TEST_START

split_assignments = modeling[
    [*identifier_columns, "split"]
].copy()

split_counts = (
    split_assignments.groupby(["city", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=SELECTED_CITIES)
    .reindex(columns=["train", "validation", "test"])
)

expected_split_counts = {
    "train": 840,
    "validation": 180,
    "test": 180,
}

for split_name, expected_count in expected_split_counts.items():
    assert split_counts[split_name].eq(expected_count).all()

split_date_summary = (
    split_assignments.groupby("split")
    .agg(
        first_target_date=("target_date", "min"),
        last_target_date=("target_date", "max"),
        rows=("target_date", "size"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

assert set(split_assignments.columns).isdisjoint(outcome_columns)
assert not split_assignments.duplicated([
    "city",
    "target_date",
]).any()

display(split_date_summary)
display(split_counts.reset_index())

print("Chronological split dates and row counts are frozen.")


## 4. Isolate Dhaka Development Data and Protect Test Outcomes

The primary experiment learns only from Dhaka. The four smaller cities are reserved for the geographic-transfer evaluation in Notebook 05.

Notebook 04 creates feature-only held-out rows for the test period and deliberately creates no test-label array. This prevents test outcomes from affecting cross-validation, model selection, threshold choice, or feature importance.


In [ ]:
dhaka_train = modeling.loc[
    modeling["city"].eq(SOURCE_CITY)
    & modeling["split"].eq("train")
].copy()

dhaka_validation = modeling.loc[
    modeling["city"].eq(SOURCE_CITY)
    & modeling["split"].eq("validation")
].copy()

dhaka_development = modeling.loc[
    modeling["city"].eq(SOURCE_CITY)
    & modeling["split"].isin(["train", "validation"])
].copy()

held_out_test = modeling.loc[
    modeling["split"].eq("test"),
    [*identifier_columns, *feature_columns],
].copy()

X_train = dhaka_train[feature_columns].copy()
y_train = dhaka_train[TARGET_COLUMN].astype(int).copy()

X_validation = dhaka_validation[feature_columns].copy()
y_validation = (
    dhaka_validation[TARGET_COLUMN]
    .astype(int)
    .copy()
)

X_development = dhaka_development[feature_columns].copy()
y_development = (
    dhaka_development[TARGET_COLUMN]
    .astype(int)
    .copy()
)

assert X_train.shape == (840, 65)
assert X_validation.shape == (180, 65)
assert X_development.shape == (1020, 65)
assert len(held_out_test) == 900
assert list(X_train.columns) == feature_columns
assert list(X_validation.columns) == feature_columns
assert list(X_development.columns) == feature_columns
assert list(held_out_test.columns) == [
    *identifier_columns,
    *feature_columns,
]
assert set(held_out_test.columns).isdisjoint(outcome_columns)
assert dhaka_train["target_date"].max() < (
    dhaka_validation["target_date"].min()
)
assert "y_test" not in globals()

development_summary = pd.DataFrame({
    "role": [
        "Dhaka training",
        "Dhaka validation",
        "Dhaka final refit",
        "Held-out test features: all cities",
    ],
    "rows": [
        len(X_train),
        len(X_validation),
        len(X_development),
        len(held_out_test),
    ],
    "labels_used_in_notebook_04": [
        True,
        True,
        True,
        False,
    ],
})

held_out_rows_by_city = (
    held_out_test.groupby("city")
    .size()
    .reindex(SELECTED_CITIES)
    .rename("held_out_feature_rows")
    .reset_index()
)

display(development_summary)
display(held_out_rows_by_city)

print("Test outcomes are excluded from the Notebook 04 workflow.")


## 5. Persistence Baseline on Dhaka Validation

Persistence predicts that tomorrow's class will equal today's class. `daily_aqi_lag1` is the AQI observed exactly one day before the target, so the baseline predicts unhealthy when that historical AQI exceeds 150.

It is evaluated before machine learning on the same 180-row Dhaka validation period. Average precision is reported, but its score is coarse because persistence provides only a hard 0/1 output rather than a continuous probability.


In [ ]:
def calculate_binary_metrics(y_true, score, prediction):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    return {
        "rows": int(len(y_true)),
        "positive_rate": float(np.mean(y_true)),
        "average_precision": float(
            average_precision_score(y_true, score)
        ),
        "recall": float(
            recall_score(y_true, prediction, zero_division=0)
        ),
        "precision": float(
            precision_score(y_true, prediction, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, prediction, zero_division=0)
        ),
        "f2": float(
            fbeta_score(
                y_true,
                prediction,
                beta=2,
                zero_division=0,
            )
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


persistence_prediction = (
    dhaka_validation["daily_aqi_lag1"]
    .gt(150)
    .astype(int)
)

persistence_score = persistence_prediction.astype(float)

persistence_metrics = calculate_binary_metrics(
    y_validation,
    persistence_score,
    persistence_prediction,
)

persistence_row = {
    "evaluation": "persistence_hard_class",
    "model": "Persistence",
    "score_type": "hard_class",
    "configuration": "daily_aqi_lag1 > 150",
    "cv_best_mean_average_precision": np.nan,
    "decision_threshold": np.nan,
    **persistence_metrics,
}

display(
    pd.DataFrame([persistence_row]).round(4)
)

print("Persistence baseline evaluated before machine learning.")


## 6. Audit the Expanding-Window Cross-Validation Folds

Hyperparameters are tuned only within the 840-row Dhaka training period using four expanding-window folds. Each validation fold occurs strictly after its corresponding fitting window.

There is no random shuffle and no resampling. Optional class weighting is fitted only inside each training fold.


In [ ]:
time_series_cv = TimeSeriesSplit(n_splits=4)
cv_fold_records = []

for fold, (fit_indices, validation_indices) in enumerate(
    time_series_cv.split(X_train),
    start=1,
):
    assert fit_indices.max() < validation_indices.min()

    fit_dates = dhaka_train.iloc[fit_indices]["target_date"]
    validation_dates = (
        dhaka_train.iloc[validation_indices]["target_date"]
    )

    assert fit_dates.max() < validation_dates.min()

    cv_fold_records.append({
        "fold": fold,
        "fit_rows": len(fit_indices),
        "fit_start": fit_dates.min(),
        "fit_end": fit_dates.max(),
        "fit_positive_rate": y_train.iloc[
            fit_indices
        ].mean(),
        "validation_rows": len(validation_indices),
        "validation_start": validation_dates.min(),
        "validation_end": validation_dates.max(),
        "validation_positive_rate": y_train.iloc[
            validation_indices
        ].mean(),
    })

cv_fold_audit = pd.DataFrame(cv_fold_records)

assert len(cv_fold_audit) == 4
assert cv_fold_audit["fit_end"].lt(
    cv_fold_audit["validation_start"]
).all()

display(cv_fold_audit.round(4))

print("Four time-ordered tuning folds verified.")


## 7. Define a Small, Justified Candidate Set

Three model families are compared:

1. **Logistic Regression** — an interpretable linear benchmark. Scaling is placed inside the pipeline, so every fold fits its own scaler using only that fold's training rows.
2. **Random Forest** — a bagged-tree model that can capture nonlinear effects and interactions.
3. **Histogram Gradient Boosting** — a compact boosted-tree model for nonlinear tabular relationships.

The grids are deliberately small. Class weighting is treated as a training-only option rather than applying SMOTE or any random temporal resampling.


In [ ]:
candidate_definitions = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=3000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]),
        "parameter_grid": {
            "model__C": [0.1, 1.0, 10.0],
            "model__class_weight": [None, "balanced"],
        },
    },
    "Random Forest": {
        "pipeline": Pipeline([
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]),
        "parameter_grid": {
            "model__max_depth": [5, 10, None],
            "model__min_samples_leaf": [2, 5],
            "model__class_weight": [None, "balanced"],
        },
    },
    "Histogram Gradient Boosting": {
        "pipeline": Pipeline([
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=200,
                    l2_regularization=1.0,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]),
        "parameter_grid": {
            "model__learning_rate": [0.05, 0.1],
            "model__max_leaf_nodes": [15, 31],
            "model__class_weight": [None, "balanced"],
        },
    },
}

configuration_counts = []

for model_name, definition in candidate_definitions.items():
    grid_size = int(np.prod([
        len(values)
        for values in definition["parameter_grid"].values()
    ]))

    configuration_counts.append({
        "model": model_name,
        "configurations": grid_size,
        "cv_fits": grid_size * time_series_cv.n_splits,
    })

configuration_counts = pd.DataFrame(configuration_counts)

assert configuration_counts["configurations"].sum() == 26
assert configuration_counts["cv_fits"].sum() == 104

display(configuration_counts)

print("Candidate configurations and tuning workload frozen.")


## 8. Tune Candidates Using Dhaka Training Data Only

Every configuration is scored by mean average precision across the four chronological folds. `GridSearchCV` refits each family's best configuration on all 840 Dhaka training rows, but the separate 180-row Dhaka validation period remains outside tuning.


In [ ]:
search_objects = {}
candidate_estimators = {}
candidate_best_parameters = {}
cv_result_frames = []

for model_name, definition in candidate_definitions.items():
    print(f"Tuning {model_name}...")

    grid_search = GridSearchCV(
        estimator=definition["pipeline"],
        param_grid=definition["parameter_grid"],
        scoring="average_precision",
        cv=time_series_cv,
        n_jobs=N_JOBS,
        refit=True,
        return_train_score=False,
        error_score="raise",
    )

    grid_search.fit(X_train, y_train)

    search_objects[model_name] = grid_search
    candidate_estimators[model_name] = (
        grid_search.best_estimator_
    )
    candidate_best_parameters[model_name] = (
        grid_search.best_params_
    )

    raw_results = pd.DataFrame(grid_search.cv_results_)
    fold_score_columns = [
        column
        for column in raw_results.columns
        if column.startswith("split")
        and column.endswith("_test_score")
    ]

    compact_results = raw_results[[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        *fold_score_columns,
        "params",
    ]].copy()

    compact_results.insert(0, "model", model_name)
    compact_results["parameters"] = compact_results[
        "params"
    ].apply(
        lambda parameters: json.dumps(
            parameters,
            sort_keys=True,
        )
    )
    compact_results = compact_results.drop(columns="params")
    cv_result_frames.append(compact_results)

    print(
        "Best CV average precision: "
        f"{grid_search.best_score_:.4f}"
    )
    print("Best parameters:", grid_search.best_params_)

cv_results = (
    pd.concat(cv_result_frames, ignore_index=True)
    .sort_values(
        ["model", "rank_test_score"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

assert len(cv_results) == 26
assert set(candidate_estimators) == set(candidate_definitions)
assert cv_results["mean_test_score"].between(0, 1).all()

display(
    cv_results.loc[
        cv_results["rank_test_score"].eq(1)
    ].round(4)
)

print("Training-only hyperparameter tuning completed.")


## 9. Select One Model Using Dhaka Validation PR-AUC

Each tuned family now generates probabilities for the same Dhaka validation rows. Model selection is based first on validation average precision, which is the threshold-independent PR-AUC summary used by this project.

An exact tie is broken by higher recall at the default 0.5 threshold and then by model simplicity: Logistic Regression, Random Forest, Histogram Gradient Boosting.


In [ ]:
simplicity_rank = {
    "Logistic Regression": 1,
    "Random Forest": 2,
    "Histogram Gradient Boosting": 3,
}

candidate_validation_probabilities = {}
candidate_validation_rows = []

for model_name, estimator in candidate_estimators.items():
    validation_probability = estimator.predict_proba(
        X_validation
    )[:, 1]
    default_prediction = (
        validation_probability >= 0.5
    ).astype(int)

    candidate_validation_probabilities[model_name] = (
        validation_probability
    )

    metrics = calculate_binary_metrics(
        y_validation,
        validation_probability,
        default_prediction,
    )

    candidate_validation_rows.append({
        "evaluation": "candidate_default_threshold",
        "model": model_name,
        "score_type": "probability",
        "configuration": json.dumps(
            candidate_best_parameters[model_name],
            sort_keys=True,
        ),
        "cv_best_mean_average_precision": float(
            search_objects[model_name].best_score_
        ),
        "decision_threshold": 0.5,
        "simplicity_rank": simplicity_rank[model_name],
        **metrics,
    })

candidate_validation_results = pd.DataFrame(
    candidate_validation_rows
)

selection_order = (
    candidate_validation_results.sort_values(
        [
            "average_precision",
            "recall",
            "simplicity_rank",
        ],
        ascending=[False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

selected_model_name = selection_order.loc[0, "model"]
selected_estimator = candidate_estimators[
    selected_model_name
]
selected_validation_probability = (
    candidate_validation_probabilities[selected_model_name]
)

assert selected_model_name in candidate_definitions
assert len(selected_validation_probability) == 180
assert np.isfinite(selected_validation_probability).all()
assert (
    (selected_validation_probability >= 0)
    & (selected_validation_probability <= 1)
).all()

display(selection_order.round(4))

print("Selected model family:", selected_model_name)
print("Selection metric: Dhaka validation average precision")


## 10. Select and Freeze a Recall-Oriented Threshold

The decision threshold is selected only for the chosen model and only from Dhaka validation probabilities.

The objective maximizes F2, which weights recall twice as strongly as precision. Ties are resolved by higher recall, then higher precision, then proximity to 0.5. The selected threshold is frozen before the final model ever reaches test data.


In [ ]:
threshold_candidates = np.unique(np.concatenate([
    np.array([0.0, 0.5, 1.0]),
    selected_validation_probability,
]))

threshold_records = []

for threshold in threshold_candidates:
    threshold_prediction = (
        selected_validation_probability >= threshold
    ).astype(int)

    metrics = calculate_binary_metrics(
        y_validation,
        selected_validation_probability,
        threshold_prediction,
    )

    threshold_records.append({
        "threshold": float(threshold),
        "distance_from_0_5": float(abs(threshold - 0.5)),
        **metrics,
    })

threshold_results = pd.DataFrame(threshold_records)

threshold_selection_order = (
    threshold_results.sort_values(
        [
            "f2",
            "recall",
            "precision",
            "distance_from_0_5",
            "threshold",
        ],
        ascending=[False, False, False, True, True],
        kind="mergesort",
    )
)

selected_threshold = float(
    threshold_selection_order.iloc[0]["threshold"]
)

selected_threshold_prediction = (
    selected_validation_probability >= selected_threshold
).astype(int)

selected_threshold_metrics = calculate_binary_metrics(
    y_validation,
    selected_validation_probability,
    selected_threshold_prediction,
)

selected_threshold_row = {
    "evaluation": "selected_f2_threshold",
    "model": selected_model_name,
    "score_type": "probability",
    "configuration": json.dumps(
        candidate_best_parameters[selected_model_name],
        sort_keys=True,
    ),
    "cv_best_mean_average_precision": float(
        search_objects[selected_model_name].best_score_
    ),
    "decision_threshold": selected_threshold,
    "simplicity_rank": simplicity_rank[selected_model_name],
    **selected_threshold_metrics,
}

validation_results = pd.concat(
    [
        pd.DataFrame([persistence_row]),
        candidate_validation_results,
        pd.DataFrame([selected_threshold_row]),
    ],
    ignore_index=True,
)

assert 0 <= selected_threshold <= 1
assert len(validation_results) == 5

display(threshold_selection_order.head(10).round(4))
display(validation_results.round(4))

print(f"Selected F2 threshold: {selected_threshold:.6f}")
print("Model family and threshold are now frozen.")


## 11. Validation-Only Diagnostics and Feature Importance

The precision–recall plot compares persistence and all candidate models on Dhaka validation data only.

Permutation importance is calculated for the selected pre-refit model using validation average precision. It estimates how much validation performance falls when a feature is shuffled. Strongly correlated lag and rolling features may share importance, so these values should be interpreted as predictive contributions rather than causal effects.


In [ ]:
fig, axis = plt.subplots(figsize=(9, 6))

baseline_precision, baseline_recall, _ = precision_recall_curve(
    y_validation,
    persistence_score,
)

axis.step(
    baseline_recall,
    baseline_precision,
    where="post",
    linewidth=2,
    linestyle="--",
    label=(
        "Persistence "
        f"(AP={persistence_metrics['average_precision']:.3f})"
    ),
)

for model_name, probability in (
    candidate_validation_probabilities.items()
):
    precision_values, recall_values, _ = precision_recall_curve(
        y_validation,
        probability,
    )
    model_ap = average_precision_score(
        y_validation,
        probability,
    )

    axis.step(
        recall_values,
        precision_values,
        where="post",
        linewidth=2,
        label=f"{model_name} (AP={model_ap:.3f})",
    )

axis.axhline(
    y_validation.mean(),
    color="gray",
    linestyle=":",
    label=(
        "Validation prevalence "
        f"({y_validation.mean():.3f})"
    ),
)
axis.set(
    title="Dhaka Validation Precision–Recall Curves",
    xlabel="Recall",
    ylabel="Precision",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
axis.grid(alpha=0.25)
axis.legend(loc="lower left")
fig.tight_layout()
fig.savefig(PR_CURVE_FILE, dpi=200, bbox_inches="tight")
plt.show()

importance_result = permutation_importance(
    selected_estimator,
    X_validation,
    y_validation,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

validation_feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": importance_result.importances_mean,
    "importance_std": importance_result.importances_std,
})

validation_feature_importance = (
    validation_feature_importance.sort_values(
        "importance_mean",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

validation_feature_importance.insert(
    0,
    "rank",
    np.arange(1, len(validation_feature_importance) + 1),
)

assert len(validation_feature_importance) == 65
assert (
    validation_feature_importance["feature"].nunique()
    == 65
)

display(validation_feature_importance.head(15).round(6))

print("Saved validation PR curve:", PR_CURVE_FILE)
print("Validation-only permutation importance completed.")


## 12. Refit the Frozen Model on Dhaka Train Plus Validation

After selection is complete, a fresh copy of the chosen pipeline is fitted on all 1,020 Dhaka development rows. The feature order, hyperparameters, and validation-selected threshold remain unchanged.

No test feature is passed to `fit`, and the refitted model is not scored in this notebook.


In [ ]:
final_model = clone(selected_estimator)
final_model.fit(X_development, y_development)

final_training_summary = pd.DataFrame({
    "item": [
        "Source city",
        "Final fitting rows",
        "Feature count",
        "First fitting target date",
        "Last fitting target date",
        "Frozen threshold",
    ],
    "value": [
        SOURCE_CITY,
        len(X_development),
        len(feature_columns),
        dhaka_development["target_date"].min().date(),
        dhaka_development["target_date"].max().date(),
        selected_threshold,
    ],
})

assert len(X_development) == 1020
assert dhaka_development["target_date"].max() == VALIDATION_END
assert dhaka_development["target_date"].max() < TEST_START
assert list(X_development.columns) == feature_columns
assert "y_test" not in globals()

display(final_training_summary)

print("Final model refitted without test evaluation.")


## 13. Save and Verify the Notebook 04 Handoff

This section saves compact evidence and the fitted model in shared storage. Large model artifacts are not committed to GitHub.

The split file contains identifiers and split membership only. The protocol tells Notebook 05 exactly which feature order, model artifact, target, threshold, and held-out dates to use. Saved artifacts are read back before the gate can pass.


In [ ]:
def make_json_ready(value):
    if isinstance(value, dict):
        return {
            str(key): make_json_ready(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [make_json_ready(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        return str(value.date())

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (float, np.floating)):
        if not np.isfinite(value):
            return None

        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    return value


split_assignments_to_save = split_assignments.copy()

for date_column in ["feature_date", "target_date"]:
    split_assignments_to_save[date_column] = (
        split_assignments_to_save[date_column]
        .dt.strftime("%Y-%m-%d")
    )

split_assignments_to_save.to_csv(
    SPLIT_ASSIGNMENTS_FILE,
    index=False,
)
cv_results.to_csv(CV_RESULTS_FILE, index=False)
validation_results.to_csv(
    VALIDATION_RESULTS_FILE,
    index=False,
)
validation_feature_importance.to_csv(
    FEATURE_IMPORTANCE_FILE,
    index=False,
)
joblib.dump(final_model, FINAL_MODEL_FILE)

split_dates = {
    "train_start": TRAIN_START,
    "train_end": TRAIN_END,
    "validation_start": VALIDATION_START,
    "validation_end": VALIDATION_END,
    "test_start": TEST_START,
    "test_end": TEST_END,
}

final_model_protocol = {
    "source_city": SOURCE_CITY,
    "transfer_cities": [
        city for city in SELECTED_CITIES if city != SOURCE_CITY
    ],
    "target_column": TARGET_COLUMN,
    "target_definition": (
        "Next calendar day's daily maximum AQI > 150"
    ),
    "sensitivity_target_column": "next_day_usg_or_worse",
    "sensitivity_target_used_for_selection": False,
    "feature_columns": feature_columns,
    "feature_count": len(feature_columns),
    "model_family": selected_model_name,
    "best_parameters": candidate_best_parameters[
        selected_model_name
    ],
    "selection_metric": "Dhaka validation average precision",
    "threshold": selected_threshold,
    "threshold_selection_rule": (
        "Maximize Dhaka validation F2; break ties by recall, "
        "precision, then proximity to 0.5"
    ),
    "split_dates": split_dates,
    "expected_rows_per_city": expected_split_counts,
    "random_state": RANDOM_STATE,
    "input_checksums": input_checksums,
    "model_artifact": FINAL_MODEL_FILE.name,
    "test_evaluation_notebook": (
        "05_evaluation_and_error_analysis.ipynb"
    ),
    "test_policy": (
        "Use the frozen model and threshold once on the held-out "
        "test period. Do not refit, retune, or change features."
    ),
}

modeling_summary = {
    "input_rows": len(modeling),
    "input_columns": 71,
    "selected_cities": SELECTED_CITIES,
    "source_city": SOURCE_CITY,
    "feature_count": len(feature_columns),
    "split_dates": split_dates,
    "split_counts_by_city": (
        split_counts.reset_index().to_dict(orient="records")
    ),
    "dhaka_train_rows": len(X_train),
    "dhaka_validation_rows": len(X_validation),
    "dhaka_final_refit_rows": len(X_development),
    "held_out_test_feature_rows": len(held_out_test),
    "cv_method": "TimeSeriesSplit with 4 expanding folds",
    "cv_scoring": "average_precision",
    "candidate_models": list(candidate_definitions),
    "candidate_configurations": int(
        configuration_counts["configurations"].sum()
    ),
    "best_parameters_by_model": candidate_best_parameters,
    "validation_results": validation_results.to_dict(
        orient="records"
    ),
    "selected_model": selected_model_name,
    "selected_threshold": selected_threshold,
    "threshold_objective": "validation F2",
    "top_validation_features": (
        validation_feature_importance.head(15)
        .to_dict(orient="records")
    ),
    "final_model_file": FINAL_MODEL_FILE.name,
    "protocol_file": FINAL_PROTOCOL_FILE.name,
    "test_labels_used_for_selection": False,
    "test_predictions_created": False,
    "test_metrics_calculated": False,
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "input_checksums": input_checksums,
}

with FINAL_PROTOCOL_FILE.open("w", encoding="utf-8") as file:
    json.dump(
        make_json_ready(final_model_protocol),
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

with MODELING_SUMMARY_FILE.open("w", encoding="utf-8") as file:
    json.dump(
        make_json_ready(modeling_summary),
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

saved_splits = pd.read_csv(
    SPLIT_ASSIGNMENTS_FILE,
    parse_dates=["feature_date", "target_date"],
)
saved_cv_results = pd.read_csv(CV_RESULTS_FILE)
saved_validation_results = pd.read_csv(
    VALIDATION_RESULTS_FILE
)
saved_importance = pd.read_csv(FEATURE_IMPORTANCE_FILE)
loaded_model = joblib.load(FINAL_MODEL_FILE)

with FINAL_PROTOCOL_FILE.open("r", encoding="utf-8") as file:
    saved_protocol = json.load(file)

with MODELING_SUMMARY_FILE.open("r", encoding="utf-8") as file:
    saved_summary = json.load(file)

assert saved_splits.shape == (6000, 4)
assert set(saved_splits.columns).isdisjoint(outcome_columns)
assert not saved_splits.duplicated([
    "city",
    "target_date",
]).any()
assert len(saved_cv_results) == 26
assert len(saved_validation_results) == 5
assert len(saved_importance) == 65
assert saved_protocol["feature_columns"] == feature_columns
assert saved_protocol["feature_count"] == 65
assert saved_protocol["model_family"] == selected_model_name
assert np.isclose(
    saved_protocol["threshold"],
    selected_threshold,
)
assert saved_summary["test_labels_used_for_selection"] is False
assert saved_summary["test_predictions_created"] is False
assert saved_summary["test_metrics_calculated"] is False
assert set(held_out_test.columns).isdisjoint(outcome_columns)
assert "y_test" not in globals()

readback_probability = loaded_model.predict_proba(
    X_development.head(2)
)[:, 1]

assert len(readback_probability) == 2
assert np.isfinite(readback_probability).all()

saved_outputs = [
    SPLIT_ASSIGNMENTS_FILE,
    CV_RESULTS_FILE,
    VALIDATION_RESULTS_FILE,
    FEATURE_IMPORTANCE_FILE,
    MODELING_SUMMARY_FILE,
    FINAL_MODEL_FILE,
    FINAL_PROTOCOL_FILE,
    PR_CURVE_FILE,
]

for output_file in saved_outputs:
    assert output_file.is_file()
    print("Saved and verified:", output_file)

print("\nNOTEBOOK 04 MODELING AND TUNING GATE PASSED")


## 14. Notebook 04 Handoff

If the previous cell prints `NOTEBOOK 04 MODELING AND TUNING GATE PASSED`, model development is complete and frozen.

Notebook 05 must then:

1. reload `modeling_dataset.csv`, the split assignments, final model, and final protocol;
2. verify all saved checksums, feature order, target, split dates, and threshold;
3. evaluate persistence and the frozen model on identical untouched test rows;
4. report Dhaka within-city performance and each smaller city's transfer performance;
5. lead with PR-AUC and recall, followed by precision, F1, confusion matrices, and prevalence;
6. analyze false negatives first and state plainly whether machine learning beat persistence.

Notebook 05 must not refit the model, retune hyperparameters, change the selected threshold, or alter the 65-feature list after viewing test outcomes.
